# LoRA Fine-Tuning with Hugging Face + W&B

This notebook demonstrates how to fine-tune an open-source LLM using:

- Hugging Face Transformers
- LoRA (Low Rank Adaptation)
- Weights & Biases experiment tracking

## Workflow

1. Authenticate Hugging Face and W&B
2. Load a pricing dataset
3. Format dataset for instruction tuning
4. Load a base model with 4-bit quantization
5. Apply LoRA adapters
6. Train and evaluate the model

The goal is to train a model that can **predict product prices from descriptions**.

In [ ]:
import os
import wandb
from huggingface_hub import login

# Hugging Face authentication
login(token=os.environ["HF_TOKEN"])

# W&B authentication
wandb.login(key=os.environ["WANDB_API_KEY"])

# Start experiment tracking
wandb.init(project="pricing-lora-training")

print("Authentication successful")

In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes wandb huggingface_hub

Load Pricing Dataset

In [ ]:
from datasets import Dataset

data = {
    "title": [
        "Wireless Bluetooth Headphones",
        "Gaming Laptop 16GB RAM",
        "USB-C Fast Charging Cable",
        "Smartphone 128GB Storage",
        "Mechanical Keyboard RGB",
        "4K Ultra HD Monitor",
        "Portable External SSD 1TB",
        "Noise Cancelling Earbuds",
        "Smartwatch Fitness Tracker",
        "Wireless Computer Mouse"
    ],
    
    "description": [
        "Over-ear headphones with noise isolation and 20-hour battery life",
        "High performance gaming laptop with RTX graphics and SSD storage",
        "Durable braided cable supporting fast charging and data transfer",
        "Latest generation smartphone with OLED display and dual cameras",
        "Mechanical keyboard with customizable RGB lighting",
        "27-inch ultra HD monitor ideal for design and gaming",
        "High speed external SSD with USB-C connectivity",
        "Compact earbuds with active noise cancellation",
        "Fitness smartwatch with heart rate and sleep monitoring",
        "Ergonomic wireless mouse with long battery life"
    ],
    
    "price": [
        79,
        1299,
        12,
        899,
        120,
        399,
        150,
        199,
        180,
        35
    ]
}

dataset = Dataset.from_dict(data)

dataset

Format + Tokenize Dataset

In [ ]:
from transformers import AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def format_example(example):
    
    prompt = f"""
Predict the price of the product.

Product: {example['title']}
Description: {example['description']}

Price:
"""

    example["text"] = prompt + str(example["price"])
    return example


dataset = dataset.map(format_example)


def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )


dataset = dataset.map(tokenize)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

dataset

Load Model + Apply LoRA

This uses parameter-efficient fine-tuning via the PEFT library.

In [ ]:
!pip install peft

In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install peft

In [ ]:
import sys
!{sys.executable} -m pip install accelerate

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

# Load base model (without device_map)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

In [ ]:
import torch
from torch.utils.data import DataLoader

# Use 'price' as labels
class PricingDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.inputs = dataset['input_ids']          # tokenized text
        self.attention_mask = dataset['attention_mask']
        self.labels = dataset['price']              # numeric price

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.inputs[idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.attention_mask[idx], dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),  # float for regression
        }

train_dataset = PricingDataset(dataset)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

# Use optimizer for LoRA parameters only
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

# Training loop (tiny demo)
for epoch in range(3):
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['input_ids']  # LM requires labels to be token IDs
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} completed, loss: {loss.item():.4f}")

# Save LoRA adapter only
model.save_pretrained("pricing-lora-model")
print("Training completed and LoRA adapter saved!")